# Preparar CSVs para Neo4j

Este notebook genera los archivos CSV necesarios para subir datos a Neo4j.

## Archivos generados:

1. **complaints_neo4j_ready.csv** - Para notebook 4.3
2. **recalls_neo4j_ready.csv** - Ya existe, generado por notebook 1.1
3. **investigations_neo4j_ready.csv** - Ya existe, generado previamente

In [ ]:
import pandas as pd
from pathlib import Path

print("="*70)
print("GENERANDO complaints_neo4j_ready.csv")
print("="*70)

In [ ]:
# Cargar datos filtrados y downsampled
df = pd.read_parquet('data/processed/complaints_filtered_downsampled.parquet', engine='pyarrow')

print(f"\n[i] Datos originales: {len(df):,} filas")
print(f"[i] Columnas disponibles: {len(df.columns)}")

In [ ]:
# Seleccionar y renombrar columnas para Neo4j
neo4j_df = pd.DataFrame({
    'complaint_id': df['CMPLID'],
    'make': df['MAKETXT'].str.strip().str.upper(),
    'model': df['MODELTXT'].str.strip().str.upper(),
    'year': df['YEARTXT'],
    'component': df['COMPDESC'],
    'description': df['CDESCR'].fillna(''),
    'open_date': df['DATEA'],
    'fail_date': df['FAILDATE'],
    'miles': df['MILES'].fillna(''),
    'city': df['CITY'].fillna(''),
    'state': df['STATE'].fillna(''),
    'crash': df['CRASH'].fillna(''),
    'fire': df['FIRE'].fillna(''),
    'injured': df['INJURED'].fillna(''),
    'deaths': df['DEATHS'].fillna(''),
    'comp_l1': df['COMP_L1']
})

# Filtrar filas con complaint_id válido
neo4j_df = neo4j_df[neo4j_df['complaint_id'].notna()].copy()

print(f"[i] Filas válidas: {len(neo4j_df):,}")

In [ ]:
# Exportar
out_dir = Path('data/neo4j/exports')
out_dir.mkdir(parents=True, exist_ok=True)

output_path = out_dir / 'complaints_neo4j_ready.csv'
neo4j_df.to_csv(output_path, index=False)

print(f"\n[OK] Archivo exportado: {output_path}")
print(f"[OK] Registros: {len(neo4j_df):,}")
print(f"[OK] Columnas: {list(neo4j_df.columns)}")

print("\nPrimeras 3 filas:")
print(neo4j_df.head(3)[['complaint_id', 'make', 'model', 'year', 'component']].to_string())